In [1]:

import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")


PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Name: Tesla T4


In [2]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

from pathlib import Path
import zipfile
import shutil

# Google Drive folder
drive_folder = Path(
    "/content/drive/MyDrive/SemFew_Final_Project"
)

# Colab working folder
work_folder = Path("/content/SemFew_Final_Project")
work_folder.mkdir(parents=True, exist_ok=True)

# Required files verify karein
required_files = [
    "CIFAR-FS.zip",
    "SemFew-main.zip",
    "ResNet-CIFAR-FS.pth"
]

for filename in required_files:
    file_path = drive_folder / filename
    assert file_path.exists(), f"Missing: {filename}"
    print("Found:", filename)

# Dataset aur GitHub code extract karein
for filename in ["SemFew-main.zip", "CIFAR-FS.zip"]:
    print("\nExtracting:", filename)

    with zipfile.ZipFile(drive_folder / filename) as z:
        z.extractall(work_folder)

    print("Extraction completed:", filename)

# Pretrained checkpoint copy karein
shutil.copy2(
    drive_folder / "ResNet-CIFAR-FS.pth",
    work_folder / "ResNet-CIFAR-FS.pth"
)

print("\nFiles in working folder:")
for item in sorted(work_folder.iterdir()):
    print(" -", item.name)

print("\nSTEP 6 COMPLETED!")


Found: CIFAR-FS.zip
Found: SemFew-main.zip
Found: ResNet-CIFAR-FS.pth

Extracting: SemFew-main.zip
Extraction completed: SemFew-main.zip

Extracting: CIFAR-FS.zip
Extraction completed: CIFAR-FS.zip

Files in working folder:
 - ResNet-CIFAR-FS.pth
 - SemFew-main
 - __MACOSX
 - cifar-fs

STEP 6 COMPLETED!


In [4]:

from pathlib import Path
from torchvision.datasets import ImageFolder
import shutil

base = Path("/content/SemFew_Final_Project")
dataset = base / "cifar-fs"
repo = base / "SemFew-main"

# Remove duplicate Jupyter checkpoint copies
extra = dataset / "train" / "apple" / ".ipynb_checkpoints"

if extra.exists():
    print("Duplicate checkpoint images:",
          len(list(extra.glob("*.png"))))
    shutil.rmtree(extra)
    print("Extra images removed from Colab copy.")

# Verify dataset splits
for split, expected_classes, expected_images in [
    ("train", 64, 38400),
    ("val", 16, 9600),
    ("test", 20, 12000)
]:
    ds = ImageFolder(str(dataset / split))

    print(f"{split}: {len(ds.classes)} classes, {len(ds)} images")

    assert len(ds.classes) == expected_classes
    assert len(ds) == expected_images

# Verify official implementation
assert (repo / "method" / "train.py").exists()
assert (repo / "method" / "SemAlign.py").exists()
assert (repo / "semantic" /
        "cifar100_semantic_clip_gpt.pth").exists()

print("\nOfficial SemFew code verified!")

# Place pretrained model in expected directory
checkpoint_dir = repo / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

shutil.copy2(
    base / "ResNet-CIFAR-FS.pth",
    checkpoint_dir / "ResNet-CIFAR-FS.pth"
)

print("Pretrained checkpoint placed successfully!")
print("\nSTEP 7 COMPLETED!")


Duplicate checkpoint images: 4
Extra images removed from Colab copy.
train: 64 classes, 38400 images
val: 16 classes, 9600 images
test: 20 classes, 12000 images

Official SemFew code verified!
Pretrained checkpoint placed successfully!

STEP 7 COMPLETED!


In [5]:

import torch
import sys
from pathlib import Path

# Project paths
base = Path("/content/SemFew_Final_Project")
repo = base / "SemFew-main"

sys.path.insert(0, str(repo))

# Original SemFew ResNet-12 architecture
from model.res12 import Res12

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# Create ResNet-12 model
model = Res12(
    avg_pool=True,
    drop_block=False
)

# Load official pretrained checkpoint
checkpoint_path = (
    repo / "checkpoints" / "ResNet-CIFAR-FS.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

# Original repository stores backbone
# parameters with an 8-character prefix
original_weights = checkpoint["params"]

model_keys = model.state_dict()

model_weights = {
    key[8:]: value
    for key, value in original_weights.items()
    if key[8:] in model_keys
}

# Ensure all required parameters are available
model.load_state_dict(model_weights, strict=True)

model = model.to(device)
model.eval()

print("Pretrained ResNet-12 loaded successfully!")

# Test model with 2 dummy RGB images
dummy_images = torch.randn(
    2, 3, 32, 32,
    device=device
)

with torch.no_grad():
    features = model(dummy_images)

print("Input shape:", dummy_images.shape)
print("Output feature shape:", features.shape)

assert features.shape == (2, 640)
assert torch.isfinite(features).all()

print("\nSTEP 8 COMPLETED!")


Using device: cuda
Pretrained ResNet-12 loaded successfully!
Input shape: torch.Size([2, 3, 32, 32])
Output feature shape: torch.Size([2, 640])

STEP 8 COMPLETED!


In [6]:

import torch
import shutil
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

# Original SemFew preprocessing
from utils import transform_val_cifar

base = Path("/content/SemFew_Final_Project")
repo = base / "SemFew-main"
dataset_path = base / "cifar-fs" / "train"

# Load 64 training classes
train_dataset = ImageFolder(
    str(dataset_path),
    transform=transform_val_cifar
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Total training images:", len(train_dataset))
print("Training classes:", len(train_dataset.classes))

# ResNet-12 from Step 8
model.eval()

num_classes = len(train_dataset.classes)
feature_dim = 640

feature_sums = torch.zeros(
    num_classes, feature_dim, dtype=torch.float64
)

class_counts = torch.zeros(
    num_classes, dtype=torch.long
)

# Extract visual features
with torch.no_grad():
    for images, labels in tqdm(
        train_loader,
        desc="Extracting ResNet-12 Features"
    ):
        images = images.to(device)

        features = model(images).cpu().double()

        feature_sums.index_add_(0, labels, features)

        class_counts.index_add_(
            0, labels,
            torch.ones_like(labels)
        )

# Compute mean class centers
assert torch.all(class_counts == 600)

mean_centers = feature_sums / class_counts.unsqueeze(1)

# Save using original class names
centers_dict = {
    class_name: mean_centers[i].float().clone()
    for i, class_name in enumerate(train_dataset.classes)
}

assert len(centers_dict) == 64

# Save in original repository's expected format
center_file = repo / "center_CIFAR-FS_resnet.pth"

torch.save({
    "mean": centers_dict,
    "center": centers_dict
}, center_file)

# Backup to Google Drive
drive_backup = Path(
    "/content/drive/MyDrive/SemFew_Final_Project"
) / "center_CIFAR-FS_resnet.pth"

shutil.copy2(center_file, drive_backup)

print("\nTotal class centers:", len(centers_dict))
print("Each center shape:", next(iter(centers_dict.values())).shape)
print("Images per class:", class_counts.unique().tolist())
print("Saved:", center_file)
print("Google Drive backup:", drive_backup)

print("\nSTEP 9 COMPLETED!")


Total training images: 38400
Training classes: 64


Extracting ResNet-12 Features:   0%|          | 0/300 [00:00<?, ?it/s]


Total class centers: 64
Each center shape: torch.Size([640])
Images per class: [600]
Saved: /content/SemFew_Final_Project/SemFew-main/center_CIFAR-FS_resnet.pth
Google Drive backup: /content/drive/MyDrive/SemFew_Final_Project/center_CIFAR-FS_resnet.pth

STEP 9 COMPLETED!


In [7]:

import torch
from pathlib import Path
from torchvision.datasets import ImageFolder
from method.SemAlign import SemAlign

base = Path("/content/SemFew_Final_Project")
repo = base / "SemFew-main"

# 1. Load official CLIP semantic embeddings
semantic_path = (
    repo / "semantic" /
    "cifar100_semantic_clip_gpt.pth"
)

semantic_data = torch.load(
    semantic_path,
    map_location="cpu",
    weights_only=True
)

semantic = {
    name: vector.float()
    for name, vector in
    semantic_data["semantic_feature"].items()
}

print("Total semantic classes:", len(semantic))
print("Semantic feature dimension:",
      next(iter(semantic.values())).shape)

# 2. Check all dataset classes have semantic features
for split in ["train", "val", "test"]:
    ds = ImageFolder(str(base / "cifar-fs" / split))
    missing = set(ds.classes) - set(semantic.keys())

    assert not missing, f"Missing semantics: {missing}"
    print(f"{split}: all {len(ds.classes)} classes matched")

assert len(semantic) == 100
assert all(v.shape == (512,) for v in semantic.values())

# 3. Verify computed class centers
center_data = torch.load(
    repo / "center_CIFAR-FS_resnet.pth",
    map_location="cpu",
    weights_only=True
)

centers = center_data["mean"]

assert len(centers) == 64
assert all(v.shape == (640,) for v in centers.values())

print("Class centers verified:", len(centers))

# 4. Initialize official SemAlign network
H = SemAlign(
    v_size=640,
    s_size=512,
    h_size=4096,
    drop=0.0
).to(device)

# 5. Test forward pass
sample_names = list(centers.keys())[:2]

text_features = torch.stack([
    semantic[name] for name in sample_names
]).to(device)

visual_features = torch.randn(2, 640, device=device)

H.eval()
with torch.no_grad():
    reconstructed = H(text_features, visual_features)

print("Visual input:", visual_features.shape)
print("Semantic input:", text_features.shape)
print("SemAlign output:", reconstructed.shape)

assert reconstructed.shape == (2, 640)
assert torch.isfinite(reconstructed).all()

print("\nSTEP 10 COMPLETED!")


Total semantic classes: 100
Semantic feature dimension: torch.Size([512])
train: all 64 classes matched
val: all 16 classes matched
test: all 20 classes matched
Class centers verified: 64
Visual input: torch.Size([2, 640])
Semantic input: torch.Size([2, 512])
SemAlign output: torch.Size([2, 640])

STEP 10 COMPLETED!


In [8]:

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm
from pathlib import Path

from utils import transform_train_cifar
from method.SemAlign import SemAlign

# Project paths
base = Path("/content/SemFew_Final_Project")
repo = base / "SemFew-main"
drive_dir = Path(
    "/content/drive/MyDrive/SemFew_Final_Project"
)

# Reproducible model initialization
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Training dataset with original augmentation
train_data = ImageFolder(
    str(base / "cifar-fs" / "train"),
    transform=transform_train_cifar
)

train_loader = DataLoader(
    train_data,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Arrange semantic features and class centers
# in exactly the same order as ImageFolder labels
semantic_matrix = torch.stack([
    semantic[name] for name in train_data.classes
]).to(device)

center_matrix = torch.stack([
    centers[name] for name in train_data.classes
]).to(device)

# Freeze pretrained ResNet-12
model.eval()
model.requires_grad_(False)

# Initialize trainable SemAlign
H = SemAlign(
    v_size=640,
    s_size=512,
    h_size=4096,
    drop=0.0
).to(device)

optimizer = torch.optim.Adam(
    H.parameters(),
    lr=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=40,
    gamma=0.1
)

print("Starting SemAlign Training...")
print("Device:", device)
print("Training images:", len(train_data))
print("Batch size:", 128)

# First epoch
H.train()

total_loss = 0.0
total_samples = 0

for images, labels in tqdm(
    train_loader,
    desc="Epoch 1/50"
):
    images = images.to(device)
    labels = labels.to(device)

    # Frozen vision encoder
    with torch.no_grad():
        visual = model(images)

    # Get matching semantic features and targets
    text = semantic_matrix[labels]
    target = center_matrix[labels]

    # Forward pass through SemAlign
    prediction = H(text, visual)

    # Original SemFew L1 alignment loss
    loss = F.l1_loss(prediction, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * len(labels)
    total_samples += len(labels)

scheduler.step()

average_loss = total_loss / total_samples

assert torch.isfinite(
    torch.tensor(average_loss)
), "Training loss is not finite!"

print("\nEpoch 1 completed!")
print("Average Training Loss:", round(average_loss, 6))

# Save checkpoint to Google Drive
checkpoint_path = (
    drive_dir / "semalign_training_checkpoint.pth"
)

torch.save({
    "epoch": 1,
    "model_state_dict": H.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "loss_history": [average_loss],
    "architecture": "ResNet12_SemAlign_CLIP",
    "dataset": "CIFAR-FS"
}, checkpoint_path)

print("Checkpoint saved:", checkpoint_path)
print("\nSTEP 11 COMPLETED!")


Starting SemAlign Training...
Device: cuda
Training images: 38400
Batch size: 128


Epoch 1/50:   0%|          | 0/300 [00:00<?, ?it/s]


Epoch 1 completed!
Average Training Loss: 0.127979
Checkpoint saved: /content/drive/MyDrive/SemFew_Final_Project/semalign_training_checkpoint.pth

STEP 11 COMPLETED!


In [9]:

import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm.auto import tqdm
import shutil
import os

# Google Drive checkpoint
drive_dir = Path(
    "/content/drive/MyDrive/SemFew_Final_Project"
)

checkpoint_path = (
    drive_dir / "semalign_training_checkpoint.pth"
)

# Resume previous training
ckpt = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

H.load_state_dict(ckpt["model_state_dict"])
optimizer.load_state_dict(ckpt["optimizer_state_dict"])
scheduler.load_state_dict(ckpt["scheduler_state_dict"])

# Match official GitHub LR schedule
scheduler.step_size = 30

start_epoch = ckpt["epoch"]
loss_history = ckpt["loss_history"]

print("Resuming from completed epoch:", start_epoch)
print("Previous loss:", loss_history[-1])
print("Learning rate:", optimizer.param_groups[0]["lr"])

model.eval()
model.requires_grad_(False)

def save_training_checkpoint(epoch, history):
    # First save locally to avoid partial Drive writes
    local_path = Path(
        "/content/semalign_checkpoint_temp.pth"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": H.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "loss_history": history,
        "architecture": "ResNet12_SemAlign_CLIP",
        "dataset": "CIFAR-FS"
    }, local_path)

    temporary_drive_path = checkpoint_path.with_suffix(
        ".pth.tmp"
    )

    shutil.copy2(local_path, temporary_drive_path)
    os.replace(temporary_drive_path, checkpoint_path)

    print(f"Checkpoint saved: Epoch {epoch}")


# Continue training to 50 epochs
for epoch in range(start_epoch, 50):

    H.train()
    total_loss = 0.0
    total_samples = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/50"
    )

    for images, labels in progress:

        images = images.to(device)
        labels = labels.to(device)

        # Frozen pretrained ResNet-12
        with torch.no_grad():
            visual = model(images)

        text = semantic_matrix[labels]
        target = center_matrix[labels]

        # SemAlign forward pass
        prediction = H(text, visual)

        # L1 alignment objective
        loss = F.l1_loss(prediction, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        total_samples += len(labels)

    scheduler.step()

    avg_loss = total_loss / total_samples

    if not torch.isfinite(torch.tensor(avg_loss)):
        raise RuntimeError("Non-finite training loss!")

    loss_history.append(avg_loss)

    print(
        f"Epoch {epoch + 1}/50 "
        f"| Average Loss: {avg_loss:.6f}"
    )

    # Backup every 5 epochs and at the end
    if (epoch + 1) % 5 == 0 or epoch == 49:
        save_training_checkpoint(
            epoch + 1,
            loss_history
        )

print("\nTRAINING COMPLETED!")
print("Total epochs:", len(loss_history))
print("Final training loss:", loss_history[-1])
print("Checkpoint:", checkpoint_path)


Resuming from completed epoch: 1
Previous loss: 0.1279791366805633
Learning rate: 0.0001


Epoch 2/50:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch 2/50 | Average Loss: 0.085960


Epoch 3/50:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch 3/50 | Average Loss: 0.065398


Epoch 4/50:   0%|          | 0/300 [00:00<?, ?it/s]

... (123 outputs left out)

Epoch 49/50 | Average Loss: 0.003819


Epoch 50/50:   0%|          | 0/300 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b0da1428900>Exception ignored in: 
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b0da1428900>Traceback (most recent call last):
Exception ignored in: 
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b0da1428900>
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b0da1428900>    Traceback (most recent call last):

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
Traceback (most recent call last):
self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__

    self._shutdown_workers()          File "/usr/local/lib/

Epoch 50/50 | Average Loss: 0.003795
Checkpoint saved: Epoch 50

TRAINING COMPLETED!
Total epochs: 50
Final training loss: 0.003795141133014113
Checkpoint: /content/drive/MyDrive/SemFew_Final_Project/semalign_training_checkpoint.pth


In [10]:

import torch
import torch.nn.functional as F
import numpy as np
import json
from pathlib import Path
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from data.samplers import CategoriesSampler
from method.SemAlign import SemAlign
from utils import transform_val_cifar

base = Path("/content/SemFew_Final_Project")
drive_dir = Path(
    "/content/drive/MyDrive/SemFew_Final_Project"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and verify completed training
ckpt = torch.load(
    drive_dir / "semalign_training_checkpoint.pth",
    map_location="cpu",
    weights_only=True
)

assert ckpt["epoch"] == 50
assert len(ckpt["loss_history"]) == 50
assert np.isfinite(ckpt["loss_history"]).all()

print("Checkpoint verified: 50 epochs completed")
print("Final loss:", ckpt["loss_history"][-1])

# Reload trained SemAlign
H = SemAlign(
    v_size=640,
    s_size=512,
    h_size=4096,
    drop=0.0
).to(device)

H.load_state_dict(ckpt["model_state_dict"])
H.eval()

# ResNet-12 from Step 8
model.eval()

# Load validation images
val_dataset = ImageFolder(
    str(base / "cifar-fs" / "val"),
    transform=transform_val_cifar
)

print("Validation classes:", len(val_dataset.classes))
print("Validation images:", len(val_dataset))

def evaluate_validation(shot):
    torch.manual_seed(13 + shot)

    sampler = CategoriesSampler(
        val_dataset.targets,
        n_batch=600,
        n_cls=5,
        n_per=shot + 15
    )

    loader = DataLoader(
        val_dataset,
        batch_sampler=sampler,
        num_workers=0,
        pin_memory=True
    )

    ks = torch.linspace(0, 1, 101, device=device)
    expected = torch.arange(
        5, device=device
    ).repeat(15)

    episode_accuracies = []
    baseline_accuracies = []

    with torch.no_grad():
        for images, labels in tqdm(
            loader,
            desc=f"Validation {shot}-shot"
        ):
            features = model(
                images.to(device)
            )

            n_support = shot * 5

            support = features[:n_support]
            query = features[n_support:]

            # Normal visual prototypes
            visual_proto = support.reshape(
                shot, 5, 640
            ).mean(dim=0)

            # Semantic features
            support_semantic = torch.stack([
                semantic[val_dataset.classes[int(label)]]
                for label in labels[:n_support]
            ]).to(device)

            # SemAlign reconstructed prototypes
            reconstructed = H(
                support_semantic, support
            )

            reconstructed_proto = reconstructed.reshape(
                shot, 5, 640
            ).mean(dim=0)

            # Evaluate all fusion values 0.00 to 1.00
            fused = (
                ks[:, None, None] *
                reconstructed_proto.unsqueeze(0)
                +
                (1 - ks[:, None, None]) *
                visual_proto.unsqueeze(0)
            )

            fused = F.normalize(fused, dim=-1)
            query_norm = F.normalize(query, dim=-1)

            logits = torch.einsum(
                "qd,knd->kqn",
                query_norm,
                fused
            )

            predictions = logits.argmax(dim=-1)

            accuracies = (
                predictions == expected.unsqueeze(0)
            ).float().mean(dim=1)

            episode_accuracies.append(
                accuracies.cpu().numpy()
            )

            # Visual-only baseline
            baseline_logits = (
                query_norm @
                F.normalize(visual_proto, dim=-1).T
            )
            baseline_predictions = baseline_logits.argmax(1)

            baseline_accuracies.append(
                (baseline_predictions == expected)
                .float().mean().item()
            )

    results = np.stack(episode_accuracies)
    means = results.mean(axis=0)

    best_idx = int(np.argmax(means))
    best_k = float(ks[best_idx].item())

    best_episode_acc = results[:, best_idx]

    accuracy = float(best_episode_acc.mean() * 100)
    ci95 = float(
        1.96 * best_episode_acc.std() /
        np.sqrt(len(best_episode_acc)) * 100
    )

    baseline = float(
        np.mean(baseline_accuracies) * 100
    )

    print(f"\n{shot}-SHOT VALIDATION RESULTS")
    print(f"Best fusion k: {best_k:.2f}")
    print(f"SemFew Accuracy: {accuracy:.2f}% ± {ci95:.2f}%")
    print(f"Visual-only Baseline: {baseline:.2f}%")

    return {
        "shot": shot,
        "best_k": round(best_k, 2),
        "validation_accuracy": accuracy,
        "ci95": ci95,
        "visual_baseline": baseline,
        "episodes": 600
    }

# Validate standard reproduction settings
result_1shot = evaluate_validation(1)
result_5shot = evaluate_validation(5)

validation_results = {
    "1shot": result_1shot,
    "5shot": result_5shot
}

# Save validation results
output_path = drive_dir / "semfew_validation_results.json"

with open(output_path, "w") as f:
    json.dump(validation_results, f, indent=4)

print("\nValidation results saved:", output_path)
print("STEP 13 COMPLETED!")


Checkpoint verified: 50 epochs completed
Final loss: 0.003795141133014113
Validation classes: 16
Validation images: 9600


Validation 1-shot:   0%|          | 0/600 [00:00<?, ?it/s]


1-SHOT VALIDATION RESULTS
Best fusion k: 0.83
SemFew Accuracy: 77.14% ± 0.74%
Visual-only Baseline: 63.38%


Validation 5-shot:   0%|          | 0/600 [00:00<?, ?it/s]


5-SHOT VALIDATION RESULTS
Best fusion k: 0.47
SemFew Accuracy: 81.20% ± 0.64%
Visual-only Baseline: 79.00%

Validation results saved: /content/drive/MyDrive/SemFew_Final_Project/semfew_validation_results.json
STEP 13 COMPLETED!


In [1]:
print("Model loaded:", "model" in globals())
print("SemAlign loaded:", "H" in globals())
print("Dataset path available:", "base" in globals())

Model loaded: False
SemAlign loaded: False
Dataset path available: False


In [2]:

import torch
import sys
import json
import zipfile
import shutil
from pathlib import Path
from google.colab import drive

# 1. Reconnect Google Drive
drive.mount("/content/drive")

drive_dir = Path(
    "/content/drive/MyDrive/SemFew_Final_Project"
)

base = Path("/content/SemFew_Final_Project")
base.mkdir(parents=True, exist_ok=True)

# 2. Restore project files
for filename, folder in [
    ("SemFew-main.zip", "SemFew-main"),
    ("CIFAR-FS.zip", "cifar-fs")
]:
    if not (base / folder).exists():
        print("Restoring:", filename)
        with zipfile.ZipFile(drive_dir / filename) as z:
            z.extractall(base)

# 3. Remove duplicate images from temporary copy
extra = base / "cifar-fs/train/apple/.ipynb_checkpoints"

if extra.exists():
    shutil.rmtree(extra)

# 4. Import official SemFew models
repo = base / "SemFew-main"
sys.path.insert(0, str(repo))

from model.res12 import Res12
from method.SemAlign import SemAlign

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 5. Restore pretrained ResNet-12
model = Res12(
    avg_pool=True,
    drop_block=False
)

backbone_checkpoint = torch.load(
    drive_dir / "ResNet-CIFAR-FS.pth",
    map_location="cpu",
    weights_only=True
)["params"]

model_keys = model.state_dict()

weights = {
    k[8:]: v
    for k, v in backbone_checkpoint.items()
    if k[8:] in model_keys
}

model.load_state_dict(weights, strict=True)
model = model.to(device)
model.eval()
model.requires_grad_(False)

print("ResNet-12 restored!")

# 6. Restore trained SemAlign
H = SemAlign(
    v_size=640,
    s_size=512,
    h_size=4096,
    drop=0.0
).to(device)

ckpt = torch.load(
    drive_dir / "semalign_training_checkpoint.pth",
    map_location="cpu",
    weights_only=True
)

H.load_state_dict(ckpt["model_state_dict"])
H.eval()

assert ckpt["epoch"] == 50

print("SemAlign restored!")
print("Completed epochs:", ckpt["epoch"])

# 7. Restore CLIP semantic embeddings
semantic_data = torch.load(
    repo / "semantic/cifar100_semantic_clip_gpt.pth",
    map_location="cpu",
    weights_only=True
)

semantic = {
    name: vector.float()
    for name, vector in
    semantic_data["semantic_feature"].items()
}

print("Semantic classes:", len(semantic))

# 8. Verify previous validation results
with open(
    drive_dir / "semfew_validation_results.json"
) as f:
    validation_results = json.load(f)

print("1-shot best k:",
      validation_results["1shot"]["best_k"])

print("5-shot best k:",
      validation_results["5shot"]["best_k"])

print("Device:", device)

print("\nRUNTIME RESTORED SUCCESSFULLY!")


Mounted at /content/drive
Restoring: SemFew-main.zip
Restoring: CIFAR-FS.zip
ResNet-12 restored!
SemAlign restored!
Completed epochs: 50
Semantic classes: 100
1-shot best k: 0.83
5-shot best k: 0.47
Device: cuda

RUNTIME RESTORED SUCCESSFULLY!


In [3]:

import torch
import torch.nn.functional as F
import numpy as np
import json
from pathlib import Path
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from data.samplers import CategoriesSampler
from utils import transform_val_cifar
from tqdm.auto import tqdm

base = Path("/content/SemFew_Final_Project")
drive_dir = Path(
    "/content/drive/MyDrive/SemFew_Final_Project"
)

# Load validation-selected fusion factors
with open(drive_dir / "semfew_validation_results.json") as f:
    validation_results = json.load(f)

# Verify the trained model checkpoint
ckpt = torch.load(
    drive_dir / "semalign_training_checkpoint.pth",
    map_location="cpu",
    weights_only=True
)

assert ckpt["epoch"] == 50

H.load_state_dict(ckpt["model_state_dict"])
H.eval()
model.eval()

# Unseen CIFAR-FS test dataset
test_dataset = ImageFolder(
    str(base / "cifar-fs" / "test"),
    transform=transform_val_cifar
)

assert len(test_dataset.classes) == 20
assert len(test_dataset) == 12000

print("Test classes:", len(test_dataset.classes))
print("Test images:", len(test_dataset))
print("Trained model verified: 50 epochs")

def evaluate_test(shot, fusion_k):
    # Fixed seed for reproducible test episodes
    torch.manual_seed(2026 + shot)

    sampler = CategoriesSampler(
        test_dataset.targets,
        n_batch=600,
        n_cls=5,
        n_per=shot + 15
    )

    test_loader = DataLoader(
        test_dataset,
        batch_sampler=sampler,
        num_workers=0,
        pin_memory=True
    )

    expected = torch.arange(
        5, device=device
    ).repeat(15)

    semfew_acc = []
    baseline_acc = []
    reconstructed_acc = []

    with torch.inference_mode():
        for images, labels in tqdm(
            test_loader,
            desc=f"Testing {shot}-shot"
        ):
            # Extract visual features
            features = model(images.to(device))

            n_support = shot * 5
            support = features[:n_support]
            query = features[n_support:]

            # Normal visual prototypes
            visual_proto = support.reshape(
                shot, 5, 640
            ).mean(dim=0)

            # CLIP semantic features
            text = torch.stack([
                semantic[test_dataset.classes[int(label)]]
                for label in labels[:n_support]
            ]).to(device)

            # Reconstructed SemAlign prototypes
            reconstructed = H(text, support)

            reconstructed_proto = reconstructed.reshape(
                shot, 5, 640
            ).mean(dim=0)

            # Paper's prototype fusion
            fused_proto = (
                fusion_k * reconstructed_proto
                + (1.0 - fusion_k) * visual_proto
            )

            query_norm = F.normalize(query, dim=-1)

            def episode_accuracy(proto):
                logits = (
                    query_norm @
                    F.normalize(proto, dim=-1).T
                )
                predicted = logits.argmax(dim=1)
                return (
                    (predicted == expected)
                    .float().mean().item()
                )

            semfew_acc.append(
                episode_accuracy(fused_proto)
            )
            baseline_acc.append(
                episode_accuracy(visual_proto)
            )
            reconstructed_acc.append(
                episode_accuracy(reconstructed_proto)
            )

    def summarize(values):
        values = np.asarray(values)
        mean = values.mean() * 100
        ci95 = (
            1.96 * values.std(ddof=1)
            / np.sqrt(len(values)) * 100
        )
        return {
            "accuracy": round(float(mean), 4),
            "ci95": round(float(ci95), 4)
        }

    result = {
        "shot": shot,
        "fusion_k": fusion_k,
        "episodes": 600,
        "semfew": summarize(semfew_acc),
        "visual_only": summarize(baseline_acc),
        "reconstructed_only": summarize(reconstructed_acc)
    }

    print(f"\n===== {shot}-SHOT TEST RESULTS =====")
    print("Fusion k:", fusion_k)

    for method in [
        "semfew",
        "visual_only",
        "reconstructed_only"
    ]:
        item = result[method]
        print(
            f"{method}: "
            f"{item['accuracy']:.2f}% "
            f"± {item['ci95']:.2f}%"
        )

    return result


# Use validation-selected k, without test tuning
k1 = validation_results["1shot"]["best_k"]
k5 = validation_results["5shot"]["best_k"]

test_1shot = evaluate_test(1, k1)
test_5shot = evaluate_test(5, k5)

results = {
    "1shot": test_1shot,
    "5shot": test_5shot,
    "paper_reference": {
        "1shot_accuracy": 83.65,
        "5shot_accuracy": 87.66
    }
}

# Save results to Google Drive
output_file = drive_dir / "semfew_final_test_results.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print("\n===== PAPER COMPARISON =====")
print("Original paper 1-shot: 83.65%")
print(
    "Our 1-shot:",
    f"{test_1shot['semfew']['accuracy']:.2f}%"
)

print("Original paper 5-shot: 87.66%")
print(
    "Our 5-shot:",
    f"{test_5shot['semfew']['accuracy']:.2f}%"
)

print("\nSaved:", output_file)
print("\nSTEP 14 COMPLETED!")


Test classes: 20
Test images: 12000
Trained model verified: 50 epochs


Testing 1-shot:   0%|          | 0/600 [00:00<?, ?it/s]


===== 1-SHOT TEST RESULTS =====
Fusion k: 0.83
semfew: 83.82% ± 0.68%
visual_only: 73.37% ± 0.89%
reconstructed_only: 80.79% ± 0.75%


Testing 5-shot:   0%|          | 0/600 [00:00<?, ?it/s]


===== 5-SHOT TEST RESULTS =====
Fusion k: 0.47
semfew: 87.55% ± 0.58%
visual_only: 86.53% ± 0.63%
reconstructed_only: 81.16% ± 0.78%

===== PAPER COMPARISON =====
Original paper 1-shot: 83.65%
Our 1-shot: 83.82%
Original paper 5-shot: 87.66%
Our 5-shot: 87.55%

Saved: /content/drive/MyDrive/SemFew_Final_Project/semfew_final_test_results.json

STEP 14 COMPLETED!


In [4]:

import torch
import torch.nn.functional as F
import numpy as np
import json
import csv
from pathlib import Path
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from data.samplers import CategoriesSampler
from utils import transform_val_cifar
from tqdm.auto import tqdm

# Use restored models from previous steps
assert "model" in globals()
assert "H" in globals()
assert "semantic" in globals()

drive_dir = Path(
    "/content/drive/MyDrive/SemFew_Final_Project"
)
base = Path("/content/SemFew_Final_Project")

model.eval()
H.eval()

def run_shot_experiment(split, shot, k_values, seed):
    ds = ImageFolder(
        str(base / "cifar-fs" / split),
        transform=transform_val_cifar
    )

    torch.manual_seed(seed)

    sampler = CategoriesSampler(
        ds.targets,
        n_batch=600,
        n_cls=5,
        n_per=shot + 15
    )

    loader = DataLoader(
        ds,
        batch_sampler=sampler,
        num_workers=0,
        pin_memory=True
    )

    ks = torch.tensor(
        k_values,
        dtype=torch.float32,
        device=device
    )

    expected = torch.arange(
        5, device=device
    ).repeat(15)

    all_accuracies = []
    all_baselines = []

    with torch.inference_mode():
        for images, labels in tqdm(
            loader,
            desc=f"{split} {shot}-shot"
        ):
            features = model(images.to(device))

            n_support = shot * 5
            support = features[:n_support]
            query = features[n_support:]

            visual_proto = support.reshape(
                shot, 5, 640
            ).mean(0)

            text = torch.stack([
                semantic[ds.classes[int(label)]]
                for label in labels[:n_support]
            ]).to(device)

            reconstructed = H(text, support)

            reconstructed_proto = reconstructed.reshape(
                shot, 5, 640
            ).mean(0)

            fused = (
                ks[:, None, None] *
                reconstructed_proto.unsqueeze(0)
                +
                (1 - ks[:, None, None]) *
                visual_proto.unsqueeze(0)
            )

            query_norm = F.normalize(query, dim=-1)

            logits = torch.einsum(
                "qd,knd->kqn",
                query_norm,
                F.normalize(fused, dim=-1)
            )

            accuracy = (
                logits.argmax(-1) ==
                expected.unsqueeze(0)
            ).float().mean(-1)

            baseline_logits = (
                query_norm @
                F.normalize(visual_proto, dim=-1).T
            )

            baseline_accuracy = (
                baseline_logits.argmax(-1) == expected
            ).float().mean()

            all_accuracies.append(
                accuracy.cpu().numpy()
            )
            all_baselines.append(
                baseline_accuracy.item()
            )

    return (
        np.stack(all_accuracies),
        np.asarray(all_baselines)
    )


def summarize(scores):
    return {
        "accuracy": round(float(scores.mean() * 100), 4),
        "ci95": round(float(
            1.96 * scores.std(ddof=1) /
            np.sqrt(len(scores)) * 100
        ), 4)
    }


# Perform additional 2-shot and 10-shot studies
new_results = {}

for shot in [2, 10]:

    print(f"\n===== {shot}-SHOT EXPERIMENT =====")

    # Choose fusion k using validation data only
    k_grid = np.linspace(0, 1, 101)

    val_scores, _ = run_shot_experiment(
        "val", shot, k_grid, seed=13 + shot
    )

    best_index = int(
        val_scores.mean(axis=0).argmax()
    )
    best_k = round(float(k_grid[best_index]), 2)

    print("Validation-selected k:", best_k)

    # Evaluate unseen test classes without tuning
    test_scores, baseline_scores = run_shot_experiment(
        "test", shot, [best_k], seed=2026 + shot
    )

    result = {
        "shot": shot,
        "fusion_k": best_k,
        "episodes": 600,
        "semfew": summarize(test_scores[:, 0]),
        "visual_only": summarize(baseline_scores)
    }

    new_results[f"{shot}shot"] = result

    print(
        "SemFew:",
        result["semfew"]["accuracy"], "%"
    )
    print(
        "Visual-only:",
        result["visual_only"]["accuracy"], "%"
    )

# Combine with existing 1-shot and 5-shot test results
with open(
    drive_dir / "semfew_final_test_results.json"
) as f:
    original_results = json.load(f)

all_results = {
    "1shot": original_results["1shot"],
    "2shot": new_results["2shot"],
    "5shot": original_results["5shot"],
    "10shot": new_results["10shot"]
}

# Save experiment results
json_path = drive_dir / "semfew_support_size_experiment.json"

with open(json_path, "w") as f:
    json.dump(all_results, f, indent=4)

csv_path = drive_dir / "semfew_support_size_results.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "Shots", "Fusion k", "SemFew Accuracy",
        "95% CI", "Visual-only Accuracy"
    ])

    for shot in [1, 2, 5, 10]:
        r = all_results[f"{shot}shot"]
        writer.writerow([
            shot,
            r["fusion_k"],
            r["semfew"]["accuracy"],
            r["semfew"]["ci95"],
            r["visual_only"]["accuracy"]
        ])

print("\n===== FINAL EXPERIMENT SUMMARY =====")

for shot in [1, 2, 5, 10]:
    r = all_results[f"{shot}shot"]
    print(
        f"{shot}-shot | "
        f"SemFew: {r['semfew']['accuracy']:.2f}% | "
        f"Visual-only: {r['visual_only']['accuracy']:.2f}%"
    )

print("\nResults saved to Google Drive")
print("STEP 15 COMPLETED!")




===== 2-SHOT EXPERIMENT =====


val 2-shot:   0%|          | 0/600 [00:00<?, ?it/s]

Validation-selected k: 0.68


test 2-shot:   0%|          | 0/600 [00:00<?, ?it/s]

SemFew: 84.7444 %
Visual-only: 79.8222 %

===== 10-SHOT EXPERIMENT =====


val 10-shot:   0%|          | 0/600 [00:00<?, ?it/s]

Validation-selected k: 0.37


test 10-shot:   0%|          | 0/600 [00:00<?, ?it/s]

SemFew: 88.5333 %
Visual-only: 88.4356 %

===== FINAL EXPERIMENT SUMMARY =====
1-shot | SemFew: 83.82% | Visual-only: 73.37%
2-shot | SemFew: 84.74% | Visual-only: 79.82%
5-shot | SemFew: 87.55% | Visual-only: 86.53%
10-shot | SemFew: 88.53% | Visual-only: 88.44%

Results saved to Google Drive
STEP 15 COMPLETED!
